In [ ]:
pip install pandas pytube moviepy

In [ ]:
import pandas as pd
from pytube import YouTube
import os
from tqdm import tqdm
from moviepy.video.io.ffmpeg_tools import ffmpeg_extract_subclip
from moviepy.editor import VideoFileClip
import cv2
import mediapipe as mp
from matplotlib import pyplot as plt

In [ ]:
import pandas as pd
from pytube import YouTube
import os
from tqdm import tqdm

def download_video(url, output_folder, video_title):
    try:
        yt = YouTube(url)
        stream = yt.streams.get_highest_resolution()
        filename = f"{video_title}.{stream.mime_type.split('/')[1]}"
        stream.download(output_path=output_folder, filename=filename)
        return True
    except Exception as e:
        print(f"Error downloading video {video_title}: {e}")
        return False

def main():
    # Read the Excel file
    df = pd.read_excel(r'ASBD-Autism-Stimming-Behavior-Datase-New.xlsx')

    # Create a folder to save the downloaded videos
    output_folder = 'downloaded_videos'
    os.makedirs(output_folder, exist_ok=True)

    # Count available videos
    available_videos_count = df['Available'].eq('Y').sum()

    # Download videos
    with tqdm(total=available_videos_count, desc="Downloading videos") as pbar:
        for index, row in df.iterrows():
            url = row['URL']
            availability = row['Available']
            video_title = row['Video']

            if availability == 'Y':
                download_success = download_video(url, output_folder, video_title)
                if download_success:
                    pbar.update(1)  # Increment progress bar if download succeeds

    print("Download complete.")

if __name__ == "__main__":
    main()


In [ ]:
def cut_into_clips(input_folder, output_folder):
    # Create the output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Iterate over each video file in the input folder
    for file_name in os.listdir(input_folder):
        if file_name.endswith('.mp4'):
            video_path = os.path.join(input_folder, file_name)
            cut_video_into_clips(video_path, output_folder)

def cut_video_into_clips(video_path, output_folder):
    try:
        # Open the video file and determine its duration
        clip = VideoFileClip(video_path)
        duration = clip.duration

        # Define the duration of each clip
        clip_duration = 3

        # Determine the number of clips to be extracted
        num_clips = int(duration / clip_duration)

        # Extract each clip
        for i in range(num_clips):
            start_time = i * clip_duration
            end_time = min(start_time + clip_duration, duration)
            output_path = os.path.join(output_folder, f"{os.path.splitext(os.path.basename(video_path))[0]}_clip_{i}.mp4")
            sub_clip = clip.subclip(start_time, end_time)
            sub_clip.write_videofile(output_path, codec="libx264", audio=False)  # Disable audio extraction
            sub_clip.close()

        # If the last clip is shorter than clip_duration, include it as well
        if duration % clip_duration != 0:
            start_time = num_clips * clip_duration
            output_path = os.path.join(output_folder, f"{os.path.splitext(os.path.basename(video_path))[0]}_clip_{num_clips}.mp4")
            sub_clip = clip.subclip(start_time, duration)
            sub_clip.write_videofile(output_path, codec="libx264", audio=False)  # Disable audio extraction
            sub_clip.close()

    except Exception as e:
        print(f"Error cutting video {video_path}: {e}")

if __name__ == "__main__":
    input_folder = r"C:\Users\nyamt\Thesis_local\Autism\DataProcessing\downloaded_videos\sliced"  # Folder containing input videos
    output_folder = r"C:\Users\nyamt\Thesis_local\Autism\DataProcessing\downloaded_videos\sliced_segments"  # Folder to save sliced videos
    cut_into_clips(input_folder, output_folder)
